# 13 — Confronto dei sei metodi del benchmark

Questo notebook **non addestra modelli**. Rilegge le predizioni dei notebook 04–12,
controlla che appartengano agli stessi casi e ricalcola metriche e matrici.
Confrontiamo Decision Tree15, Random Forest balanced, GaussianNB, SVM RBF, MLP
e K-Means con voto; l'albero15balanced è una variante aggiuntiva separata.

Cinque split stratificati80/20 seed 1–5 da KDDTrain+, tutte le 41 feature. Validation
di sviluppo, non test ufficiale;31 U2R distinti complessivi,10 per singola run.
Le ripetizioni si sovrappongono: media e DS non dimostrano significatività statistica.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

project_root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.targets import FAMILY_ORDER

saved_models = {
    'decision_tree': 'rare_class_reference',
    'random_forest_balanced': 'random_forest_balanced',
    'gaussian_naive_bayes': 'gaussian_naive_bayes',
    'svm_rbf': 'svm_rbf',
    'neural_network': 'neural_network',
    'kmeans': 'kmeans',
    'decision_tree_balanced': 'decision_tree_balanced',
}
model_labels = {
    'decision_tree': 'Decision Tree15', 'random_forest_balanced': 'Random Forest balanced',
    'gaussian_naive_bayes': 'Gaussian Naive Bayes', 'svm_rbf': 'SVM RBF',
    'neural_network': 'Rete neurale MLP100', 'kmeans': 'K-Means300 + voto',
    'decision_tree_balanced': 'Variante: Decision Tree15 balanced',
}
primary_models = list(saved_models)[:6]

## 1. Verificare il protocollo del riferimento

Servono gli artefatti locali già prodotti:05 congela il riferimento04,06–07
producono le varianti,09–12 i nuovi metodi. Non si riaddestra nulla qui.
L'hash lega le misure al CSV locale; il confronto verifica poi indici e risposte
run per run. Rifiutiamo formati sconosciuti, identità errate o protocolli discordanti.

In [ ]:
reference_path = project_root / 'artifacts/rare_class_reference'
reference = json.loads((reference_path / 'results.json').read_text(encoding='utf-8'))
source_sha256 = hashlib.sha256((project_root / 'data/raw/KDDTrain+.csv').read_bytes()).hexdigest()
seeds = [1, 2, 3, 4, 5]
assert reference['source_sha256'] == source_sha256 and reference['seeds'] == seeds
assert reference['family_order'] == list(FAMILY_ORDER) and reference['feature_count'] == 41
assert reference['validation_size'] == 0.2
with np.load(reference_path / 'predictions.npz', allow_pickle=False) as arrays:
    reference_arrays = {key: arrays[key].copy() for key in arrays.files}

In [ ]:
paired_predictions = {}
metadata_by_model = {}
for model_id, folder in saved_models.items():
    path = project_root / 'artifacts' / folder
    metadata = json.loads((path / 'results.json').read_text(encoding='utf-8'))
    expected_schema = 1 if model_id in {'decision_tree', 'decision_tree_balanced', 'random_forest_balanced'} else 2
    if metadata.get('schema_version', 1) != expected_schema:
        raise ValueError(f'Schema sconosciuto per {model_id}')
    actual_identity = metadata.get('model_id', 'decision_tree' if model_id == 'decision_tree' else None)
    if actual_identity != model_id:
        raise ValueError(f'Identità discordante per {model_id}')
    expected = {'source_sha256': source_sha256, 'seeds': seeds, 'family_order': list(FAMILY_ORDER),
                'feature_count': 41, 'validation_size': 0.2, 'splits': reference['splits']}
    if any(metadata.get(key) != value for key, value in expected.items()):
        raise ValueError(f'Protocollo discordante per {model_id}')
    paired_predictions[model_id] = {}
    with np.load(path / 'predictions.npz', allow_pickle=False) as arrays:
        for run, seed in enumerate(seeds, start=1):
            indices = arrays[f'run_{run:02d}_validation_indices']
            truth = arrays[f'run_{run:02d}_truth']
            suffix = 'depth_15' if expected_schema == 1 else 'predicted'
            predicted = arrays[f'run_{run:02d}_{suffix}']
            if not np.array_equal(indices, reference_arrays[f'run_{run:02d}_validation_indices']):
                raise ValueError(f'Indici discordanti: {model_id}, seed{seed}')
            if not np.array_equal(truth, reference_arrays[f'run_{run:02d}_truth']):
                raise ValueError(f'Target discordanti: {model_id}, seed{seed}')
            if predicted.shape != truth.shape or not np.isin(predicted, FAMILY_ORDER).all():
                raise ValueError(f'Predizioni non valide: {model_id}, seed{seed}')
            counts = confusion_matrix(truth, predicted, labels=FAMILY_ORDER)
            recorded_counts = (metadata['confusion_counts'][run - 1] if expected_schema == 2
                               else metadata['confusion_counts_by_depth']['15'][run - 1])
            if not np.array_equal(counts, recorded_counts):
                raise ValueError(f'Conteggi JSON e predizioni NPZ discordanti per {model_id}, seed{seed}')
            paired_predictions[model_id][seed] = {'truth': truth.copy(), 'predicted': predicted.copy()}
    metadata_by_model[model_id] = metadata

## 2. Ricalcolare metriche e matrici dalle predizioni

Tutte le cinque classi restano nelle metriche anche se non vengono mai predette.
Normalizziamo ciascuna matrice per classe vera, poi mediamo le cinque matrici.
Calcoliamo precision, recall e F1 per singola run prima di fare le medie.

In [ ]:
overall_rows, class_rows = [], []
comparison_matrices, comparison_std_matrices = {}, {}
for model_id, runs in paired_predictions.items():
    normalized = []
    for seed, values in runs.items():
        truth, predicted = values['truth'], values['predicted']
        report = classification_report(truth, predicted, labels=FAMILY_ORDER, output_dict=True, zero_division=0)
        counts = confusion_matrix(truth, predicted, labels=FAMILY_ORDER)
        normalized.append(confusion_matrix(truth, predicted, labels=FAMILY_ORDER, normalize='true'))
        overall_rows.append({'model': model_id, 'seed': seed, 'accuracy': accuracy_score(truth, predicted),
                             'macro_f1': report['macro avg']['f1-score'], 'weighted_f1': report['weighted avg']['f1-score']})
        for i, family in enumerate(FAMILY_ORDER):
            support, correct = int(counts[i].sum()), int(counts[i, i])
            class_rows.append({'model': model_id, 'seed': seed, 'famiglia': family,
                'support': support, 'corretti': correct, 'errori': support - correct,
                'precision': report[family]['precision'], 'recall': report[family]['recall'],
                'f1': report[family]['f1-score'], 'quota_errori': (support - correct) / support,
                'falsi_positivi': int(counts[:, i].sum() - correct)})
    comparison_matrices[model_id] = np.stack(normalized).mean(axis=0)
    comparison_std_matrices[model_id] = np.stack(normalized).std(axis=0, ddof=1)
overall_results = pd.DataFrame(overall_rows)
class_results = pd.DataFrame(class_rows)
overall_summary = overall_results.groupby('model', sort=False)[['accuracy', 'macro_f1', 'weighted_f1']].agg(['mean', 'std', 'min', 'max'])
class_summary = class_results.groupby(['model', 'famiglia'], sort=False)[
    ['precision', 'recall', 'f1', 'quota_errori', 'falsi_positivi']].agg(['mean', 'std', 'min', 'max'])
display(overall_summary.reindex(primary_models).round(6).rename(index=model_labels))
display(overall_summary.loc[['decision_tree_balanced']].round(6).rename(index=model_labels))

## 3. Guardare tutte le classi, poi dettagliare R2L e U2R

La quota di errore è1−recall: risponde a «quanti attacchi reali perdiamo?».
La precision risponde a «quanti allarmi di questa classe sono corretti?».
Accuracy e weighted F1 possono nascondere le famiglie con pochi record.

In [ ]:
error_percent = class_results.groupby(['model', 'famiglia'], sort=False)['quota_errori'].mean().unstack() * 100
display(error_percent.reindex(index=primary_models, columns=FAMILY_ORDER).rename(index=model_labels).round(2))
rare_summary = class_summary.loc[(slice(None), ['R2L', 'U2R']), :]
display(rare_summary.round(4))
display(class_results.loc[class_results['famiglia'].eq('U2R'),
    ['model', 'seed', 'support', 'corretti', 'errori', 'precision', 'recall', 'falsi_positivi']].round(4))

## 4. Confusion matrix normalizzate, sulla stessa scala

Righe=classe vera, colonne=classe predetta. Diagonale=recall. Ogni pannello è la
media delle cinque prove; le deviazioni standard restano nei CSV. Le matrici
misurano errori fra classi, non correlazioni fra feature.

In [ ]:
comparison_figure, axes = plt.subplots(3, 2, figsize=(14, 15), layout='constrained')
for axis, model_id in zip(axes.flat, primary_models):
    ConfusionMatrixDisplay(comparison_matrices[model_id], display_labels=FAMILY_ORDER).plot(
        ax=axis, cmap='Blues', values_format='.3f', colorbar=False, im_kw={'vmin': 0, 'vmax': 1})
    axis.set(title=model_labels[model_id], xlabel='Classe predetta', ylabel='Classe vera')
comparison_figure.colorbar(axes.flat[0].images[0], ax=list(axes.flat), shrink=0.6)
comparison_figure.suptitle('Sei metodi: media delle matrici normalizzate sui cinque split80/20')
plt.show()

## 5. Differenze abbinate rispetto alla Random Forest già misurata

Ogni differenza confronta modello e RF sullo stesso seed e sulla stessa classe.
Per recall/precision/F1 valori positivi indicano un aumento; per errore e FP
valori negativi indicano una diminuzione. Nessun test di significatività: queste
run condividono dati e non sono osservazioni indipendenti.

In [ ]:
reference_classes = class_results.loc[class_results['model'].eq('random_forest_balanced')]
paired = class_results.loc[~class_results['model'].eq('random_forest_balanced')].merge(
    reference_classes, on=['seed', 'famiglia'], suffixes=('_model', '_rf'), validate='many_to_one')
paired_deltas = paired[['model_model', 'seed', 'famiglia']].rename(columns={'model_model': 'model'})
for metric in ['precision', 'recall', 'f1', 'quota_errori', 'falsi_positivi']:
    paired_deltas[metric] = paired[f'{metric}_model'] - paired[f'{metric}_rf']
display(paired_deltas.groupby(['model', 'famiglia'], sort=False)[
    ['precision', 'recall', 'f1', 'quota_errori', 'falsi_positivi']].agg(['mean', 'std', 'min', 'max']).round(4))

## 6. Salvare il confronto e distinguere le configurazioni

Albero15 senza pesi, variante albero15balanced e RF100 alberi/profondità15balanced:
numeriche non scalate. NB/SVM/MLP/K-Means: numeriche standardizzate, senza pesi
aggiunti. Tutti usano one-hot appreso sul train e tutte le 41 feature. Questi risultati
confrontano configurazioni dichiarate; non attribuiscono ogni differenza al solo
algoritmo e non stabiliscono una graduatoria universale dei metodi.

In [ ]:
artifacts = project_root / 'artifacts/benchmark_comparison'
artifacts.mkdir(parents=True, exist_ok=True)
overall_results.to_csv(artifacts / 'run_metrics.csv', index=False)
class_results.to_csv(artifacts / 'class_metrics.csv', index=False)
overall_summary.to_csv(artifacts / 'overall_summary.csv')
class_summary.to_csv(artifacts / 'class_summary.csv')
error_percent.reindex(columns=FAMILY_ORDER).to_csv(artifacts / 'class_error_percent.csv')
paired_deltas.to_csv(artifacts / 'paired_deltas_vs_rf.csv', index=False)
comparison_figure.savefig(artifacts / 'confusion_matrices.png', dpi=150)
for model_id in saved_models:
    pd.DataFrame(comparison_matrices[model_id], index=FAMILY_ORDER, columns=FAMILY_ORDER).to_csv(artifacts / f'{model_id}_confusion_mean.csv')
    pd.DataFrame(comparison_std_matrices[model_id], index=FAMILY_ORDER, columns=FAMILY_ORDER).to_csv(artifacts / f'{model_id}_confusion_std.csv')
manifest = {'source_sha256': source_sha256, 'seeds': seeds, 'validation_size': .2,
            'feature_count': 41, 'family_order': list(FAMILY_ORDER), 'models': saved_models,
            'primary_models': primary_models, 'new_fits': 0, 'std_ddof': 1,
            'paired_indices_and_truth_checked': True, 'metrics_recomputed_from_predictions': True}
(artifacts / 'verification.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

## 7. Limiti e interpretazione

Solo31 U2R distinti vengono riutilizzati nelle cinque validation: la loro variabilità
non scompare ripetendo il training. Restano duplicati e overlap descritti in
DATA_AUDIT.md. Il test ufficiale, con attacchi nuovi, è un esperimento successivo.

Completiamo la **copertura dei sei tipi di modello** del paper, non il benchmark
ufficiale: split, encoding, selezione e tuning sono differenti. La TabellaIV usa
un altro protocollo; non confrontare direttamente i suoi punteggi con questi.

L'interpretazione con i numeri misurati viene aggiunta dopo l'audit indipendente.

## Esito misurato e ricontrollato

| Modello | Accuracy | Macro F1 media ± DS | Weighted F1 |
|---|---|---|---|
| Decision Tree15 | 99.73% | 0.8754 ± 0.0550 | 0.9973 |
| Random Forest balanced | 99.83% | 0.9200 ± 0.0312 | 0.9982 |
| Gaussian Naive Bayes | 74.18% | 0.4344 ± 0.0383 | 0.7834 |
| SVM RBF | 99.32% | 0.8406 ± 0.0352 | 0.9931 |
| Rete neurale MLP100 | 99.68% | 0.8921 ± 0.0377 | 0.9968 |
| K-Means300 + voto | 98.10% | 0.7593 ± 0.0481 | 0.9808 |

| Modello | Precision U2R | Recall U2R | Errore U2R | FP U2R/run | Precision R2L | Recall R2L | Errore R2L |
|---|---|---|---|---|---|---|---|
| Decision Tree15 | 49.62% | 40.00% | 60.00% | 3.2 | 95.98% | 93.47% | 6.53% |
| Random Forest balanced | 78.34% | 60.00% | 40.00% | 2.2 | 94.83% | 97.09% | 2.91% |
| Gaussian Naive Bayes | 0.80% | 88.00% | 12.00% | 1089.0 | 6.06% | 48.94% | 51.06% |
| SVM RBF | 62.33% | 26.00% | 74.00% | 1.6 | 88.45% | 84.52% | 15.48% |
| Rete neurale MLP100 | 65.12% | 48.00% | 52.00% | 2.6 | 96.14% | 92.26% | 7.74% |
| K-Means300 + voto | 11.43% | 8.00% | 92.00% | 1.6 | 78.98% | 76.28% | 23.72% |

La RF balanced conserva il miglior macro F1 medio tra queste configurazioni.
Rimane però un errore medio U2R del 40%: il problema delle classi rare non è risolto.
La rete migliora macro F1 e recall U2R rispetto a SVM, ma non supera la RF;
la precision R2L leggermente maggiore accompagna un recall inferiore.
NB riconosce molti U2R generando però 1089 falsi positivi U2R/run: quel recall da solo
non indica un buon classificatore. K-Means riconosce solo l'8% degli U2R:
ogni training elegge almeno un cluster U2R, ma quattro validation su cinque
non contengono alcun U2R riconosciuto correttamente.
Le statistiche sono descrittive su holdout sovrapposti, non prove di significatività.


### Controllo conclusivo del confronto

I conteggi ricalcolati da ogni NPZ devono coincidere con quelli registrati nel
JSON del modello. Il controllo intercetta anche predizioni incompatibili che
condividono indici e target. Correzione verificata con nove regressioni e confronto
reale rieseguito senza fit; nessun risultato misurato cambiato. Suite completa:67 test.
